# ⚡ CyberAgent v3.5 — Fast Fine-Tuning on Google Colab (Qwen 2.5 3B)
Train your autonomous cybersecurity agent with **Unsloth + QLoRA** in **15 minutes** on a free Colab GPU (Tesla T4).

### 🎯 What this notebook does:
1. Installs ultra-fast Unsloth engine (uses 80% less VRAM, 5x faster training).
2. Loads `Qwen 2.5 3B-Instruct` in 4-bit quantization.
3. Loads your CyberAgent 41-tools ethical hacking & tool-calling dataset.
4. Trains LoRA adapters with `SFTTrainer`.
5. Tests the fine-tuned model live.
6. Exports directly to **GGUF (Q4_K_M)** so you can run it locally in **Ollama**!


In [ ]:
# Step 1: Check GPU availability
!nvidia-smi


In [ ]:
# Step 2: Install Unsloth & dependencies (super fast 5x training)
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes


In [ ]:
# Step 3: Load Qwen 2.5 3B-Instruct in 4-bit
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detect (Float16 on T4, Bfloat16 on Ampere)
load_in_4bit = True

print('[*] Loading Qwen 2.5 3B model...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Qwen2.5-3B-Instruct',
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
print('✔ Base model loaded successfully!')


In [ ]:
# Step 4: Add LoRA Adapters for Function Calling & Cyber Knowledge
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407,
)
print('✔ LoRA adapters attached!')


In [ ]:
# Step 5: Upload Dataset (cyberagent_master_multiturn_dataset.jsonl)
import os
from google.colab import files

possible_files = ['cyberagent_master_multiturn_dataset.jsonl', 'cyber_agent_finetune_dataset.jsonl']
dataset_file = next((f for f in possible_files if os.path.exists(f)), None)

if not dataset_file:
    print('Please upload cyberagent_master_multiturn_dataset.jsonl (or cyber_agent_finetune_dataset.jsonl):')
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith('.jsonl'):
            dataset_file = fname
            break

print(f'[+] Using dataset file: {dataset_file}')


In [ ]:
# Step 6: Format Dataset with ChatML Template
from datasets import load_dataset

raw_dataset = load_dataset('json', data_files=dataset_file, split='train')
print(f'Total training samples: {len(raw_dataset)}')

def format_chatml(batch):
    formatted_texts = []
    for messages in batch['messages']:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
        formatted_texts.append(text)
    return {'text': formatted_texts}

formatted_dataset = raw_dataset.map(format_chatml, batched=True)
print('Sample Formatted Entry:')
print(formatted_dataset[0]['text'][:350] + '...')


In [ ]:
# Step 7: Train the Model with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = 'text',
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 80,  # ~10-15 mins on Colab T4
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 5,
        optim = 'adamw_8bit',
        weight_decay = 0.01,
        lr_scheduler_type = 'linear',
        seed = 3407,
        output_dir = 'outputs',
    ),
)

print('🚀 Starting fine-tuning...')
trainer_stats = trainer.train()
print('✔ Training complete!')


In [ ]:
# Step 8: Live Test inside Colab
FastLanguageModel.for_inference(model)

messages = [
    {'role': 'system', 'content': 'You are CyberAgent v3.5, an elite autonomous AI Ethical Hacker and Penetration Tester.'},
    {'role': 'user', 'content': 'bizmatepro.xyz এর whois এবং আইপি ডিটেইলস বের করো'}
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = 'pt'
).to('cuda')

outputs = model.generate(input_ids = inputs, max_new_tokens = 256, use_cache = True)
response = tokenizer.batch_decode(outputs)
print('\n--- Model Output ---')
print(response[0][len(tokenizer.decode(inputs[0])):])


In [ ]:
# Step 9: Export to GGUF (Q4_K_M) for Ollama
print('[*] Exporting to GGUF (Q4_K_M) format for Ollama...')
model.save_pretrained_gguf('cyberagent_qwen3b_gguf', tokenizer, quantization_method = 'q4_k_m')
print('✔ GGUF Export completed in folder: cyberagent_qwen3b_gguf/')
!ls -lh cyberagent_qwen3b_gguf/


In [ ]:
# Step 10: Download GGUF to your PC for Ollama
import glob
from google.colab import files

gguf_files = glob.glob('cyberagent_qwen3b_gguf/*.gguf')
if gguf_files:
    print(f'Downloading {gguf_files[0]} ...')
    files.download(gguf_files[0])
else:
    print('GGUF file not found in cyberagent_qwen3b_gguf/')
